# 🚀 เริ่มต้น — ตรวจเครื่องมือ + รู้จักโจทย์

ยินดีต้อนรับสู่เวิร์กช็อป **Advanced Data Engineering & Applied Analytics**

**แผนวันนี้**
- 🌅 **เช้า** — ใช้ Python ทำ ETL → โหลดเข้า **MSSQL เป็น Star Schema** (จบกระบวนการ ETL)
- 🌆 **บ่าย** — ต่อ **Power BI** เข้า MSSQL → สร้าง **Dashboard**

> โน้ตบุ๊กนี้แค่เช็คว่าเครื่องมือพร้อม + รู้จักข้อมูล แล้วไปต่อ `01_explore`

## 1) ตั้ง prefix ของคุณ
ทุกคนใช้ login เดียวกัน (`deadmin`) เขียนลง `de_loan_dw` แต่ใช้ **`TEAM_ID` นำหน้าชื่อตาราง** กันชนกัน
(เช่น `team01_fact_loan`, `team02_fact_loan`)
👉 แก้ `TEAM_ID` เป็นชื่อ/ทีมของคุณ

In [3]:
TEAM_ID = "Somboon_TEAM"   # 👈 เปลี่ยนเป็นชื่อ/ทีมของคุณ (ใช้นำหน้าชื่อตาราง)
print("Somboon_TEAM:", TEAM_ID)

Somboon_TEAM: Somboon_TEAM


## 2) ตรวจว่ามี library ครบ

In [1]:
import sys
import pandas as pd
import sqlalchemy

print("Python     :", sys.version.split()[0])
print("pandas     :", pd.__version__)
print("sqlalchemy :", sqlalchemy.__version__)

try:
    import pymssql
    print("pymssql    :", pymssql.__version__, "→ ต่อ MSSQL ได้ ✅")
except Exception as e:
    print("pymssql    : ยังไม่มี (จะใช้ตอนบ่าย/โหลด MSSQL) —", e)

Python     : 3.13.13
pandas     : 3.0.3
sqlalchemy : 2.0.50
pymssql    : 2.3.13 → ต่อ MSSQL ได้ ✅


## 3) รู้จักข้อมูล — Lending Club Loans

ข้อมูลคือ **สินเชื่อ (loan)** ของ Lending Club แต่ละแถว = 1 สินเชื่อ มีข้อมูลเช่น
วงเงิน, ดอกเบี้ย, เกรดความเสี่ยง (A–G), วัตถุประสงค์, รัฐ, และ **สถานะ** (จ่ายครบ / เบี้ยวหนี้)

**คำถามธุรกิจที่เราจะตอบด้วย dashboard:**
- เกรดไหน **เบี้ยวหนี้ (default) สูง** เมื่อเทียบกับดอกเบี้ยที่คิด? → ตั้งราคาถูกไหม
- สินเชื่อกลุ่มไหน **กำไร/ขาดทุน**?
- แนวโน้มยอดปล่อยกู้ + อัตราเบี้ยวหนี้เป็นยังไง?

## 4) เป้าหมายของวันนี้ (ภาพรวม)

```
loan_sample.csv ──► Python ETL ──► Star Schema ──► MSSQL (schema ของทีม) ──► Power BI
 (ข้อมูลดิบ)        clean+transform   dims + fact                              dashboard
```

## 5) แอบดูข้อมูลสัก 5 แถว

In [4]:
import pandas as pd
df = pd.read_csv("../data/loan_sample.csv")
print("ขนาดข้อมูล (แถว, คอลัมน์):", df.shape)
df.head()

ขนาดข้อมูล (แถว, คอลัมน์): (5020, 22)


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,issue_d,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high,total_pymnt,total_rec_prncp,recoveries
0,1000000,20000,20000,60 months,15.37%,479.69,C,C1,1 year,OWN,...,Sep-2016,Fully Paid,moving,TX,5.11,705,709,28345.89,20000.00,0.0
1,1000001,25000,25000,36 months,13.44%,847.66,C,C1,< 1 year,OWN,...,Mar-2017,Fully Paid,other,CO,16.86,760,764,29185.45,25000.00,0.0
2,1000002,8000,8000,60 months,12.06%,178.20,B,B5,2 years,OWN,...,Jan-2016,Current,car,GA,16.63,780,784,4587.99,3432.84,0.0
3,1000003,5000,5000,36 months,5.97%,152.04,A,A3,7 years,MORTGAGE,...,Jul-2016,Fully Paid,medical,NY,22.84,720,724,5426.27,5000.00,0.0
4,1000004,25000,25000,36 months,18.78%,913.62,D,D5,1 year,OWN,...,Jan-2018,In Grace Period,other,WA,27.69,690,694,15039.05,11431.21,0.0


---
✅ ถ้ารันด้านบนได้หมด = พร้อมแล้ว! ไปต่อที่ **`01_explore.ipynb`** เพื่อสำรวจข้อมูลและออกแบบ Star Schema